# Tau Slot Threshold Sweep

Notebook autocontido para calibracao de `tau_slot` usando top-1 de slots em cache.

Objetivos desta versao:
- usar apenas datasets publicos ja presentes no projeto
- amostrar sentencas por `(dataset_name, semantic_intent)`
- manter questions e hypothesis templates escopados pelo dataset de cada exemplo
- rodar a parte pesada de slot inference uma unica vez
- salvar, por sentenca, o maior span e score por slot em formato wide
- fazer o sweep offline de `tau_slot` sobre esse cache local


In [ ]:
import sys
import json
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path) -> Path:
    resolved = start.resolve()
    for candidate in [resolved, *resolved.parents]:
        if (candidate / 'src').exists() and (candidate / 'datasets').exists():
            return candidate
    raise RuntimeError(f'Could not find repository root starting from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f'REPO_ROOT = {REPO_ROOT}')

from src.core.pipeline import initialize_nlu
from src.common.text_utils import (
    add_row_scores_masked,
    build_alias_collapsed_grouping_view,
    find_all_matching_substrings,
    process_until_no_more_groups,
    remove_duplicates,
    remove_duplicates_and_substrings,
)
from src.data_tools.batch_eval_utils import (
    _batch_qa_for_intent,
    _batch_retrieval_titles,
    _build_slot_score_matrices_batch,
    _extract_entities_from_doc,
    _extract_slot_tokens_from_doc,
    _filter_doctitles_for_intent,
    _project_slot_score_columns,
    build_dataset_and_semantic_views,
    build_semantic_intent_slot_dict,
    export_evaluation_csvs,
    load_semantic_alias_registry,
    validate_registry_against_metadata,
)
from src.data_tools.batch_metrics_utils import (
    build_evaluation_examples,
    compute_slot_conll_reports,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)


In [ ]:
RUN_ID = datetime.now().strftime('tau_slot_%Y%m%d_%H%M%S')
STUDY_TIMESTAMP = datetime.now().isoformat(timespec='seconds')

THRESHOLD_ROOT = REPO_ROOT / 'threshold_evaluation'
RUN_DIR = THRESHOLD_ROOT / 'runs' / RUN_ID
ARTIFACTS_DIR = RUN_DIR / 'artifacts'
EVALUATION_DIR = RUN_DIR / 'evaluation'

for path in (THRESHOLD_ROOT, RUN_DIR, ARTIFACTS_DIR, EVALUATION_DIR):
    path.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = REPO_ROOT / 'datasets' / 'metadata' / 'semantic_alias_registry.json'
SAMPLING_SEED = 42
SAMPLES_PER_INTENT = 25
TAU_GRID = [round(step * 0.05, 2) for step in range(0, 41)]
SLOT_DELIMITER = ' || '
DEVICE = 0
SPACY_BATCH_SIZE = 64
SPACY_N_PROCESS = 1
QA_BATCH_SIZE = 16
SLOT_SENTENCE_BATCH_SIZE = 8
NLI_BATCH_SIZE = 16
REUSE_CACHED_SLOT_TOP1_SCORES = True
FLOAT_FORMAT = '%.10f'
COVERAGE_CONSTRAINT = 0.98

SLOT_TOP1_CACHE_PATH = ARTIFACTS_DIR / 'public_calibration_pool_slot_top1_scores.csv'
SAMPLED_DATASET_PATH = ARTIFACTS_DIR / 'sampled_dataset_view.csv'
SAMPLED_SEMANTIC_PATH = ARTIFACTS_DIR / 'sampled_semantic_view.csv'
SAMPLE_COUNTS_PATH = ARTIFACTS_DIR / 'sample_counts_by_dataset_intent.csv'
SWEEP_OVERALL_PATH = ARTIFACTS_DIR / 'tau_slot_sweep_overall.csv'
SWEEP_BY_DATASET_PATH = ARTIFACTS_DIR / 'tau_slot_sweep_by_dataset.csv'
SWEEP_BY_INTENT_PATH = ARTIFACTS_DIR / 'tau_slot_sweep_by_intent.csv'
SWEEP_BY_SLOT_PATH = ARTIFACTS_DIR / 'tau_slot_sweep_by_slot.csv'
BEST_TAU_PATH = ARTIFACTS_DIR / 'best_tau_slot_selection.json'
BEST_PREDICTIONS_PATH = ARTIFACTS_DIR / 'best_tau_slot_predictions.csv'
STUDY_CONFIG_PATH = ARTIFACTS_DIR / 'study_config.json'

config_preview = {
    'RUN_ID': RUN_ID,
    'RUN_DIR': str(RUN_DIR),
    'SAMPLING_SEED': SAMPLING_SEED,
    'SAMPLES_PER_INTENT': SAMPLES_PER_INTENT,
    'TAU_GRID': TAU_GRID,
    'DEVICE': DEVICE,
    'SLOT_TOP1_CACHE_PATH': str(SLOT_TOP1_CACHE_PATH),
}
display(pd.DataFrame([config_preview]))


In [ ]:
PUBLIC_CALIBRATION_DATASET_PATHS = [
    REPO_ROOT / 'datasets/snips/raw/train_AddToPlaylist_full.json',
    REPO_ROOT / 'datasets/snips/raw/train_BookRestaurant_full.json',
    REPO_ROOT / 'datasets/snips/raw/train_GetWeather_full.json',
    REPO_ROOT / 'datasets/snips/processed/clean/train_PlayMusic_full_clean.json',
    REPO_ROOT / 'datasets/snips/processed/clean/train_RateBook_full_clean.json',
    REPO_ROOT / 'datasets/snips/processed/clean/train_SearchCreativeWork_full_clean.json',
    REPO_ROOT / 'datasets/snips/processed/clean/train_SearchScreeningEvent_full_clean.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_attraction__find_attraction.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_hospital__find_hospital.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_hotel__find_hotel.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_restaurant__find_restaurant.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_taxi__find_taxi.json',
    REPO_ROOT / 'datasets/multiwoz/processed/with_slots/slots_train__find_train.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/abbreviation_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/aircraft_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/airfare_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/airline_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/flight_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/flight_time_min50.json',
    REPO_ROOT / 'datasets/atis/processed/filtered/min50/ground_service_min50.json',
]

paths_df = pd.DataFrame({'dataset_path': [str(path) for path in PUBLIC_CALIBRATION_DATASET_PATHS]})
display(paths_df)


In [ ]:
registry = load_semantic_alias_registry(REGISTRY_PATH)
validate_registry_against_metadata(registry)

normalized_public_paths = [str(path).replace('\\', '/') for path in PUBLIC_CALIBRATION_DATASET_PATHS]
if any('/field_operations/' in path.lower() for path in normalized_public_paths):
    raise ValueError('field_operations nao deve entrar no pool publico de calibracao.')

dataset_view, semantic_view = build_dataset_and_semantic_views(
    PUBLIC_CALIBRATION_DATASET_PATHS,
    registry=registry,
    slot_delimiter=SLOT_DELIMITER,
)

dataset_names = sorted(dataset_view['dataset_name'].drop_duplicates().tolist())
intent_slot_dict_by_dataset = {
    dataset_name: build_semantic_intent_slot_dict([dataset_name], registry=registry)
    for dataset_name in dataset_names
}

missing_rows = []
for _, row in semantic_view.iterrows():
    dataset_name = str(row['dataset_name'])
    semantic_intent = str(row['semantic_intent'])
    if semantic_intent not in intent_slot_dict_by_dataset.get(dataset_name, {}):
        missing_rows.append((dataset_name, semantic_intent, row['source_file'], row['example_id']))
if missing_rows:
    raise KeyError(f'Rows with semantic intents missing from their dataset-scoped slot dict: {missing_rows[:10]}')

dataset_scoped_summary = pd.DataFrame(
    [
        {
            'dataset_name': dataset_name,
            'semantic_intent_count': len(intent_slot_dict),
            'semantic_slot_count': sum(len(intent_cfg.get('slots', {})) for intent_cfg in intent_slot_dict.values()),
        }
        for dataset_name, intent_slot_dict in intent_slot_dict_by_dataset.items()
    ]
).sort_values('dataset_name', kind='mergesort').reset_index(drop=True)

print(f'dataset_view rows = {len(dataset_view)}')
print(f'semantic_view rows = {len(semantic_view)}')
print(f'dataset_names = {dataset_names}')
display(dataset_scoped_summary)
display(semantic_view[['dataset_name', 'dataset_intent', 'semantic_intent', 'sentence']].head(10))


In [ ]:
MERGE_KEYS = ['example_id', 'source_file', 'dataset_name', 'dataset_intent', 'semantic_intent']
BASE_COLUMNS = ['example_id', 'dataset_name', 'source_file', 'dataset_intent', 'semantic_intent', 'sentence', 'has_repeated_slot_value']

def safe_div(numerator, denominator):
    return 0.0 if denominator == 0 else float(numerator) / float(denominator)

def to_serializable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: to_serializable(inner_value) for key, inner_value in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if not isinstance(value, (list, tuple, dict, str)):
        try:
            if pd.isna(value):
                return None
        except (TypeError, ValueError):
            pass
    return value

def strip_accents(text):
    return ''.join(ch for ch in unicodedata.normalize('NFD', str(text)) if unicodedata.category(ch) != 'Mn')

def norm_text(text):
    clean = strip_accents('' if text is None else text).lower().strip()
    return ' '.join(clean.split())

def normalized_slot_json(slot_json):
    if not slot_json:
        return {}
    payload = json.loads(slot_json) if isinstance(slot_json, str) else dict(slot_json)
    normalized = {}
    for slot_name, values in payload.items():
        if isinstance(values, str):
            values = [values]
        cleaned_values = sorted({norm_text(value) for value in values if norm_text(value)})
        if cleaned_values:
            normalized[str(slot_name)] = cleaned_values
    return normalized

def original_slot_json(slot_json):
    if not slot_json:
        return {}
    payload = json.loads(slot_json) if isinstance(slot_json, str) else dict(slot_json)
    normalized = {}
    for slot_name, values in payload.items():
        if isinstance(values, str):
            values = [values]
        cleaned_values = [str(value) for value in values if str(value).strip()]
        if cleaned_values:
            normalized[str(slot_name)] = cleaned_values
    return normalized

def json_dumps(payload):
    return json.dumps(to_serializable(payload), ensure_ascii=False, sort_keys=True)

def batched(items, batch_size):
    if batch_size <= 0:
        raise ValueError('batch_size must be a positive integer.')
    for start in range(0, len(items), batch_size):
        yield start, items[start : start + batch_size]

def stratified_sample_semantic_view(semantic_view, sample_size=25, seed=42):
    sort_cols = ['dataset_name', 'semantic_intent', 'source_file', 'example_id']
    ordered = semantic_view.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
    sampled_groups = []
    stats_rows = []
    for (dataset_name, semantic_intent), group_df in ordered.groupby(['dataset_name', 'semantic_intent'], sort=False, dropna=False):
        available_count = len(group_df)
        sampled_count = min(sample_size, available_count)
        shortfall_count = max(sample_size - available_count, 0)
        sampled_group = group_df.copy() if sampled_count == available_count else group_df.sample(n=sampled_count, random_state=seed)
        sampled_group = sampled_group.sort_values(sort_cols, kind='mergesort')
        sampled_groups.append(sampled_group)
        stats_rows.append({'dataset_name': dataset_name, 'semantic_intent': semantic_intent, 'available_count': int(available_count), 'sampled_count': int(sampled_count), 'shortfall_count': int(shortfall_count)})
    sampled_view = pd.concat(sampled_groups, ignore_index=True) if sampled_groups else ordered.iloc[0:0].copy()
    sampled_view = sampled_view.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)
    stats_view = pd.DataFrame(stats_rows).sort_values(['dataset_name', 'semantic_intent'], kind='mergesort').reset_index(drop=True)
    return sampled_view, stats_view

def format_slot_column(raw_slot, semantic_slot):
    return f'{raw_slot} ({semantic_slot})'

def slot_map_for_row(row, registry):
    return registry[str(row['dataset_name'])]['intents'][str(row['dataset_intent'])].get('slot_map', {})

def build_raw_slot_specs(dataset_view, registry):
    specs = []
    seen_columns = set()
    unique_intents = dataset_view[['dataset_name', 'dataset_intent']].drop_duplicates()
    for _, row in unique_intents.iterrows():
        dataset_name = str(row['dataset_name'])
        dataset_intent = str(row['dataset_intent'])
        slot_map = registry[dataset_name]['intents'][dataset_intent].get('slot_map', {})
        for raw_slot, semantic_slot in slot_map.items():
            gold_column = format_slot_column(raw_slot, semantic_slot)
            if gold_column in seen_columns:
                continue
            seen_columns.add(gold_column)
            specs.append({'raw_slot': raw_slot, 'semantic_slot': semantic_slot, 'gold_column': gold_column, 'pred_column': f'pred__{gold_column}', 'score_column': f'score__{gold_column}'})
    return specs

def clean_cell_value(value):
    if value is None or pd.isna(value):
        return ''
    return str(value)

def initialize_slot_top1_cache_frame(sampled_dataset_view, raw_slot_specs, registry):
    rows = []
    for _, row in sampled_dataset_view.iterrows():
        output_row = {column: row.get(column, '') for column in BASE_COLUMNS}
        row_slot_map = slot_map_for_row(row, registry)
        for spec in raw_slot_specs:
            output_row[spec['gold_column']] = ''
            output_row[spec['pred_column']] = ''
            output_row[spec['score_column']] = ''
        for raw_slot, semantic_slot in row_slot_map.items():
            gold_column = format_slot_column(raw_slot, semantic_slot)
            output_row[gold_column] = clean_cell_value(row.get(raw_slot, ''))
        output_row['gold_slots_dataset_json'] = row.get('gold_slots_dataset_json', '{}')
        output_row['gold_slots_semantic_json'] = row.get('gold_slots_semantic_json', '{}')
        output_row['pred_intent'] = row.get('semantic_intent', '')
        output_row['pred_intent_score'] = 1.0
        output_row['slot_eval_intent'] = row.get('semantic_intent', '')
        output_row['pred_slots_semantic_json'] = '{}'
        output_row['pred_slots_score_json'] = '{}'
        output_row['candidate_spans_json'] = '[]'
        output_row['qa_scores_json'] = '{}'
        output_row['qa_spans_json'] = '{}'
        rows.append(output_row)
    slot_columns = []
    for spec in raw_slot_specs:
        slot_columns.extend([spec['gold_column'], spec['pred_column'], spec['score_column']])
    final_columns = BASE_COLUMNS + slot_columns + ['gold_slots_dataset_json', 'gold_slots_semantic_json', 'pred_intent', 'pred_intent_score', 'slot_eval_intent', 'pred_slots_semantic_json', 'pred_slots_score_json', 'candidate_spans_json', 'qa_scores_json', 'qa_spans_json']
    return pd.DataFrame(rows).reindex(columns=final_columns)

def top_slot_predictions_from_frame(df_slot_score):
    slot_values = {}
    slot_scores = {}
    if df_slot_score is None or getattr(df_slot_score, 'empty', True) or len(df_slot_score.columns) == 0:
        for slot_name in list(getattr(df_slot_score, 'index', [])):
            slot_scores[str(slot_name)] = 0.0
        return slot_values, slot_scores
    for slot_name in df_slot_score.index:
        row_scores = df_slot_score.loc[slot_name].fillna(0.0).astype(float)
        if row_scores.empty:
            slot_scores[str(slot_name)] = 0.0
            continue
        top_span = str(row_scores.idxmax())
        top_score = float(row_scores.loc[top_span])
        if top_span:
            slot_values[str(slot_name)] = [top_span]
        slot_scores[str(slot_name)] = top_score
    return slot_values, slot_scores

def fill_wide_predictions(result_view, row_index, row_slot_map, slot_values, slot_scores):
    for raw_slot, semantic_slot in row_slot_map.items():
        gold_column = format_slot_column(raw_slot, semantic_slot)
        pred_column = f'pred__{gold_column}'
        score_column = f'score__{gold_column}'
        values = slot_values.get(semantic_slot, [])
        result_view.at[row_index, pred_column] = values[0] if values else ''
        result_view.at[row_index, score_column] = float(slot_scores.get(semantic_slot, 0.0) or 0.0)

def run_batch_slot_top1_scores(nlu_processor, sampled_dataset_view, raw_slot_specs, intent_slot_dict_by_dataset, registry):
    result_view = initialize_slot_top1_cache_frame(sampled_dataset_view, raw_slot_specs, registry)
    group_sizes = {}
    start_slots = time.perf_counter()

    for (dataset_name, slot_intent), group_frame in result_view.groupby(['dataset_name', 'slot_eval_intent'], sort=False, dropna=False):
        dataset_name = str(dataset_name)
        slot_intent = str(slot_intent)
        group_sizes[f'{dataset_name}::{slot_intent}'] = len(group_frame)
        intent_slot_dict = intent_slot_dict_by_dataset[dataset_name]
        if slot_intent not in intent_slot_dict:
            raise KeyError(f'Semantic intent {slot_intent!r} not found for dataset {dataset_name!r}.')

        sentences = group_frame['sentence'].astype(str).tolist()
        docs = list(nlu_processor.models.nlp.pipe(sentences, batch_size=SPACY_BATCH_SIZE, n_process=SPACY_N_PROCESS))
        retrieval_titles = _batch_retrieval_titles(nlu_processor, sentences)
        qa_results = _batch_qa_for_intent(nlu_processor, slot_intent, sentences, intent_slot_dict, batch_size=QA_BATCH_SIZE)

        final_spans_per_sentence = []
        per_sentence_payloads = []
        for doc, sentence, titles, qa_bundle in zip(docs, sentences, retrieval_titles, qa_results):
            pos_tokens = _extract_slot_tokens_from_doc(doc, slot_intent, intent_slot_dict)
            entities = _extract_entities_from_doc(doc)
            matching_titles = find_all_matching_substrings(titles, sentence)
            matching_titles = _filter_doctitles_for_intent(nlu_processor.models.nlp, matching_titles, slot_intent, intent_slot_dict)
            relevant_spans = remove_duplicates_and_substrings(pos_tokens + matching_titles)
            qa_answers, qa_slot_scores, qa_slot_spans = qa_bundle
            final_spans = remove_duplicates(relevant_spans + entities + qa_answers)
            final_spans_per_sentence.append(final_spans)
            per_sentence_payloads.append((qa_slot_scores, qa_slot_spans))

        group_indices = list(group_frame.index)
        for start, _batch_indices in batched(group_indices, SLOT_SENTENCE_BATCH_SIZE):
            batch_indices = group_indices[start : start + SLOT_SENTENCE_BATCH_SIZE]
            batch_sentences = result_view.loc[batch_indices, 'sentence'].astype(str).tolist()
            batch_spans = final_spans_per_sentence[start : start + SLOT_SENTENCE_BATCH_SIZE]
            batch_payloads = per_sentence_payloads[start : start + SLOT_SENTENCE_BATCH_SIZE]
            batch_scores = _build_slot_score_matrices_batch(nlu_processor, slot_intent, batch_sentences, batch_spans, intent_slot_dict, batch_size=NLI_BATCH_SIZE)

            for row_index, sentence, df_slot_score, qa_payload in zip(batch_indices, batch_sentences, batch_scores, batch_payloads):
                qa_slot_scores, qa_slot_spans = qa_payload
                grouping_df_slot_score, _alias_group_map = build_alias_collapsed_grouping_view(df_slot_score, sentence)
                selected_tokens = process_until_no_more_groups(grouping_df_slot_score, threshold=None, fallback_top_k=0)
                final_df_slot_score = _project_slot_score_columns(grouping_df_slot_score, selected_tokens)
                final_df_slot_score = add_row_scores_masked(final_df_slot_score, qa_slot_scores, qa_slot_spans)
                slot_values, slot_scores = top_slot_predictions_from_frame(final_df_slot_score)
                row_slot_map = slot_map_for_row(result_view.loc[row_index], registry)
                fill_wide_predictions(result_view, row_index, row_slot_map, slot_values, slot_scores)
                result_view.at[row_index, 'pred_slots_semantic_json'] = json_dumps(slot_values)
                result_view.at[row_index, 'pred_slots_score_json'] = json_dumps(slot_scores)
                result_view.at[row_index, 'candidate_spans_json'] = json_dumps(selected_tokens)
                result_view.at[row_index, 'qa_scores_json'] = json_dumps(qa_slot_scores)
                result_view.at[row_index, 'qa_spans_json'] = json_dumps(qa_slot_spans)

    elapsed = time.perf_counter() - start_slots
    stats = {'slot_seconds': elapsed, 'n_rows': len(result_view), 'rows_per_second': safe_div(len(result_view), elapsed), 'group_sizes': group_sizes, 'grouping_threshold': None}
    return result_view, stats

def load_slot_top1_cache(path):
    cache = pd.read_csv(path, encoding='utf-8-sig', keep_default_na=False, na_filter=False)
    for column in [column for column in cache.columns if column.startswith('score__')]:
        cache[column] = pd.to_numeric(cache[column], errors='coerce').fillna(0.0)
    cache['pred_intent_score'] = pd.to_numeric(cache.get('pred_intent_score', 1.0), errors='coerce').fillna(1.0)
    return cache


In [ ]:
raw_slot_specs = build_raw_slot_specs(dataset_view, registry)
raw_slot_specs_df = pd.DataFrame(raw_slot_specs)
display(raw_slot_specs_df.head(20))

sampled_semantic_view, sample_counts_by_dataset_intent = stratified_sample_semantic_view(
    semantic_view,
    sample_size=SAMPLES_PER_INTENT,
    seed=SAMPLING_SEED,
)

sample_keys = sampled_semantic_view[MERGE_KEYS].copy()
sample_keys['_sample_order'] = np.arange(len(sample_keys))

sampled_dataset_view = (
    dataset_view
    .merge(sample_keys, on=MERGE_KEYS, how='inner')
    .sort_values('_sample_order', kind='mergesort')
    .drop(columns=['_sample_order'])
    .reset_index(drop=True)
)
sampled_semantic_view = sampled_semantic_view.reset_index(drop=True)

if len(sampled_dataset_view) != len(sampled_semantic_view):
    raise ValueError(f'sampled_dataset_view ({len(sampled_dataset_view)}) and sampled_semantic_view ({len(sampled_semantic_view)}) must have the same number of rows.')

sampled_dataset_view.to_csv(SAMPLED_DATASET_PATH, index=False, encoding='utf-8-sig')
sampled_semantic_view.to_csv(SAMPLED_SEMANTIC_PATH, index=False, encoding='utf-8-sig')
sample_counts_by_dataset_intent.to_csv(SAMPLE_COUNTS_PATH, index=False, encoding='utf-8-sig')

written_gold_paths = export_evaluation_csvs(dataset_view=sampled_dataset_view, evaluation_root=EVALUATION_DIR)

print(f'sampled_dataset_view rows = {len(sampled_dataset_view)}')
print(f'sampled_semantic_view rows = {len(sampled_semantic_view)}')
display(sample_counts_by_dataset_intent)


In [ ]:
nlu_processor = initialize_nlu(device=DEVICE, tau_intent=0.2, tau_slots=0.5)

model_snapshot = {
    'nli_model_name': getattr(nlu_processor.models, 'nli_model_name', None),
    'qa_model_name': getattr(nlu_processor.models, 'qa_model_name', None),
    'spacy_model': getattr(nlu_processor.models, 'spacy_model', None),
    'index_path': getattr(nlu_processor.models, 'index_path', None),
    'device': getattr(nlu_processor.models, 'device', None),
}
display(pd.DataFrame([model_snapshot]))


In [ ]:
if REUSE_CACHED_SLOT_TOP1_SCORES and SLOT_TOP1_CACHE_PATH.exists():
    slot_top1_cache = load_slot_top1_cache(SLOT_TOP1_CACHE_PATH)
    slot_inference_stats = {'reused_cache': True, 'n_rows': len(slot_top1_cache), 'slot_seconds': 0.0, 'grouping_threshold': None}
else:
    slot_top1_cache, slot_inference_stats = run_batch_slot_top1_scores(
        nlu_processor,
        sampled_dataset_view,
        raw_slot_specs,
        intent_slot_dict_by_dataset,
        registry,
    )
    slot_top1_cache.to_csv(SLOT_TOP1_CACHE_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)

written_top1_prediction_paths = export_evaluation_csvs(results_view=slot_top1_cache, evaluation_root=EVALUATION_DIR)

print(f'Cached slot top-1 predictions at: {SLOT_TOP1_CACHE_PATH}')
print('Slot inference stats:', json.dumps(to_serializable(slot_inference_stats), indent=2, ensure_ascii=False))
display(slot_top1_cache.head(10))


In [ ]:
def slot_names_for_row(row):
    dataset_name = str(row['dataset_name'])
    semantic_intent = str(row['semantic_intent'])
    return list(intent_slot_dict_by_dataset[dataset_name][semantic_intent].get('slots', {}).keys())

def top_value_for_slot(pred_slot_values, semantic_slot):
    values = pred_slot_values.get(semantic_slot, [])
    return values[0] if values else ''

def top_score_for_slot(pred_slot_scores, semantic_slot):
    try:
        return float(pred_slot_scores.get(semantic_slot, 0.0) or 0.0)
    except Exception:
        return 0.0

def build_thresholded_prediction_view(slot_top1_cache, tau_slot):
    thresholded = slot_top1_cache.copy()
    pred_columns = [column for column in thresholded.columns if column.startswith('pred__') and ' (' in column]
    score_columns = [column for column in thresholded.columns if column.startswith('score__')]
    for column in pred_columns:
        thresholded[column] = ''
    for column in score_columns:
        thresholded[column] = ''

    for row_index, row in thresholded.iterrows():
        try:
            top_scores = json.loads(row.get('pred_slots_score_json', '{}') or '{}')
        except Exception:
            top_scores = {}
        top_values = original_slot_json(row.get('pred_slots_semantic_json', '{}'))
        row_slot_map = slot_map_for_row(row, registry)
        pred_dict = {}
        score_dict = {}
        for raw_slot, semantic_slot in row_slot_map.items():
            top_span = top_value_for_slot(top_values, semantic_slot)
            top_score = top_score_for_slot(top_scores, semantic_slot)
            if top_span and top_score >= float(tau_slot):
                pred_dict[semantic_slot] = [top_span]
                score_dict[semantic_slot] = top_score
                gold_column = format_slot_column(raw_slot, semantic_slot)
                thresholded.at[row_index, f'pred__{gold_column}'] = top_span
                thresholded.at[row_index, f'score__{gold_column}'] = top_score
        thresholded.at[row_index, 'pred_slots_semantic_json'] = json_dumps(pred_dict)
        thresholded.at[row_index, 'pred_slots_score_json'] = json_dumps(score_dict)
    return thresholded

def build_tau_decision_frames(slot_top1_cache, tau_slot):
    decision_rows = []
    sentence_rows = []
    for row_number, row in slot_top1_cache.iterrows():
        gold_original = original_slot_json(row.get('gold_slots_semantic_json', '{}'))
        gold_normalized = normalized_slot_json(row.get('gold_slots_semantic_json', '{}'))
        top_values = original_slot_json(row.get('pred_slots_semantic_json', '{}'))
        try:
            top_scores = json.loads(row.get('pred_slots_score_json', '{}') or '{}')
        except Exception:
            top_scores = {}
        pred_dict = {}
        row_base = {key: row.get(key, '') for key in MERGE_KEYS + ['sentence']}
        row_base['row_number'] = int(row_number)

        for semantic_slot in slot_names_for_row(row):
            gold_values = gold_original.get(semantic_slot, [])
            gold_values_norm = set(gold_normalized.get(semantic_slot, []))
            top_span = top_value_for_slot(top_values, semantic_slot)
            top_score = top_score_for_slot(top_scores, semantic_slot)
            predicted = bool(top_span) and top_score >= float(tau_slot)
            value_match = predicted and norm_text(top_span) in gold_values_norm
            has_gold = bool(gold_values)
            if predicted:
                pred_dict[semantic_slot] = [top_span]
            decision = dict(row_base)
            decision.update(
                {
                    'semantic_slot': semantic_slot,
                    'has_gold': has_gold,
                    'predicted': predicted,
                    'top_span': top_span,
                    'top_score': top_score,
                    'gold_values_json': json_dumps(gold_values),
                    'value_match': value_match,
                    'value_tp': int(value_match),
                    'value_fp': int(predicted and not value_match),
                    'value_fn': int(has_gold and not value_match),
                }
            )
            decision_rows.append(decision)

        pred_normalized = {slot_name: sorted({norm_text(value) for value in values if norm_text(value)}) for slot_name, values in pred_dict.items()}
        pred_normalized = {slot_name: values for slot_name, values in pred_normalized.items() if values}
        sentence_row = dict(row_base)
        sentence_row.update(
            {
                'tau_slot': float(tau_slot),
                'pred_slots_semantic_json_tau': json_dumps(pred_dict),
                'slot_exact_match': pred_normalized == gold_normalized,
            }
        )
        sentence_rows.append(sentence_row)

    decisions = pd.DataFrame(decision_rows)
    sentence_metrics = pd.DataFrame(sentence_rows)
    if not decisions.empty:
        decisions['tau_slot'] = float(tau_slot)
    return decisions, sentence_metrics

def summarize_value_decisions(decisions, sentence_metrics=None):
    total_slot_decisions = int(len(decisions))
    covered_count = int(decisions['predicted'].sum()) if total_slot_decisions else 0
    gold_slot_count = int(decisions['has_gold'].sum()) if total_slot_decisions else 0
    gold_covered_count = int((decisions['has_gold'] & decisions['predicted']).sum()) if total_slot_decisions else 0
    value_tp = int(decisions['value_tp'].sum()) if total_slot_decisions else 0
    value_fp = int(decisions['value_fp'].sum()) if total_slot_decisions else 0
    value_fn = int(decisions['value_fn'].sum()) if total_slot_decisions else 0
    value_precision = safe_div(value_tp, value_tp + value_fp)
    value_recall = safe_div(value_tp, value_tp + value_fn)
    value_f1 = safe_div(2 * value_precision * value_recall, value_precision + value_recall) if value_precision + value_recall else 0.0
    rows = int(decisions[MERGE_KEYS].drop_duplicates().shape[0]) if total_slot_decisions else 0
    output = {
        'rows': rows,
        'total_slot_decisions': total_slot_decisions,
        'covered_count': covered_count,
        'coverage': safe_div(covered_count, total_slot_decisions),
        'gold_slot_count': gold_slot_count,
        'gold_covered_count': gold_covered_count,
        'gold_slot_coverage': safe_div(gold_covered_count, gold_slot_count),
        'value_tp': value_tp,
        'value_fp': value_fp,
        'value_fn': value_fn,
        'value_precision': value_precision,
        'value_recall': value_recall,
        'value_f1': value_f1,
        'selective_accuracy': safe_div(value_tp, covered_count),
    }
    if sentence_metrics is not None:
        exact_count = int(sentence_metrics['slot_exact_match'].sum()) if not sentence_metrics.empty else 0
        output['slot_exact_match_count'] = exact_count
        output['slot_exact_match'] = safe_div(exact_count, len(sentence_metrics))
    return output

def empty_conll_metrics(include_rows=True):
    output = {
        'tp': 0,
        'fp': 0,
        'fn': 0,
        'precision': 0.0,
        'recall': 0.0,
        'f1': 0.0,
    }
    if include_rows:
        output['rows'] = 0
    return output

def conll_metrics_from_summary(slot_conll_summary, include_rows=True):
    if slot_conll_summary is None or slot_conll_summary.empty:
        return empty_conll_metrics(include_rows=include_rows)
    row = slot_conll_summary.iloc[0]
    output = {
        'tp': int(row['tp']),
        'fp': int(row['fp']),
        'fn': int(row['fn']),
        'precision': float(row['precision']),
        'recall': float(row['recall']),
        'f1': float(row['f1']),
    }
    if include_rows:
        output['rows'] = int(row['rows'])
    return output

def summarize_conll_examples(evaluation_examples):
    slot_conll_summary, _slot_conll_by_label, _slot_conll_by_intent = compute_slot_conll_reports(evaluation_examples)
    return conll_metrics_from_summary(slot_conll_summary, include_rows=True)

def summarize_conll_slot(evaluation_examples, semantic_slot):
    _slot_conll_summary, slot_conll_by_label, _slot_conll_by_intent = compute_slot_conll_reports(evaluation_examples)
    if slot_conll_by_label is None or slot_conll_by_label.empty:
        return empty_conll_metrics(include_rows=False)
    conll_slot_name = str(semantic_slot).replace(' ', '_')
    slot_rows = slot_conll_by_label.loc[slot_conll_by_label['semantic_slot'].astype(str) == conll_slot_name]
    if slot_rows.empty:
        return empty_conll_metrics(include_rows=False)
    row = slot_rows.iloc[0]
    return {
        'tp': int(row['tp']),
        'fp': int(row['fp']),
        'fn': int(row['fn']),
        'precision': float(row['precision']),
        'recall': float(row['recall']),
        'f1': float(row['f1']),
    }

def filter_frame_by_group(frame, group_cols, group_key):
    filtered = frame
    for column, value in zip(group_cols, group_key):
        if column in filtered.columns:
            filtered = filtered.loc[filtered[column] == value]
    return filtered

def summarize_grouped_tau(decisions, sentence_metrics, evaluation_examples, tau_slot, group_cols, include_sentence_exact=True):
    rows = []
    for group_key, group_df in decisions.groupby(group_cols, sort=False, dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        row = {'tau_slot': float(tau_slot)}
        row.update({column: value for column, value in zip(group_cols, group_key)})
        sentence_group = None
        if include_sentence_exact:
            sentence_group = sentence_metrics.copy()
            for column, value in zip(group_cols, group_key):
                if column in sentence_group.columns:
                    sentence_group = sentence_group.loc[sentence_group[column] == value]

        value_metrics = summarize_value_decisions(group_df, sentence_group if include_sentence_exact else None)
        if 'semantic_slot' in group_cols:
            conll_group_cols = [column for column in group_cols if column != 'semantic_slot']
            conll_group_key = [value for column, value in zip(group_cols, group_key) if column != 'semantic_slot']
            conll_examples = filter_frame_by_group(evaluation_examples, conll_group_cols, conll_group_key)
            conll_metrics = summarize_conll_slot(conll_examples, row['semantic_slot'])
        else:
            conll_examples = filter_frame_by_group(evaluation_examples, group_cols, group_key)
            conll_metrics = summarize_conll_examples(conll_examples)

        row.update(value_metrics)
        row.update(conll_metrics)
        rows.append(row)
    return pd.DataFrame(rows)

overall_rows = []
by_dataset_frames = []
by_intent_frames = []
by_slot_frames = []

previous_coverage = None
for tau_slot in TAU_GRID:
    thresholded_view = build_thresholded_prediction_view(slot_top1_cache, tau_slot=tau_slot)
    evaluation_examples = build_evaluation_examples(thresholded_view, registry=registry)
    decisions, sentence_metrics = build_tau_decision_frames(slot_top1_cache, tau_slot=tau_slot)
    value_metrics = summarize_value_decisions(decisions, sentence_metrics)
    conll_metrics = summarize_conll_examples(evaluation_examples)
    overall_metrics = {**value_metrics, **conll_metrics}
    current_coverage = overall_metrics['coverage']
    if previous_coverage is not None and current_coverage > previous_coverage + 1e-12:
        raise ValueError('Coverage increased as tau_slot increased; sweep cache is not monotonic.')
    previous_coverage = current_coverage
    overall_rows.append({'tau_slot': float(tau_slot), **overall_metrics})
    by_dataset_frames.append(summarize_grouped_tau(decisions, sentence_metrics, evaluation_examples, tau_slot, ['dataset_name']))
    by_intent_frames.append(summarize_grouped_tau(decisions, sentence_metrics, evaluation_examples, tau_slot, ['dataset_name', 'semantic_intent']))
    by_slot_frames.append(summarize_grouped_tau(decisions, sentence_metrics, evaluation_examples, tau_slot, ['dataset_name', 'semantic_intent', 'semantic_slot'], include_sentence_exact=False))

tau_slot_sweep_overall = pd.DataFrame(overall_rows).sort_values('tau_slot', kind='mergesort').reset_index(drop=True)
tau_slot_sweep_by_dataset = pd.concat(by_dataset_frames, ignore_index=True).sort_values(['tau_slot', 'dataset_name'], kind='mergesort').reset_index(drop=True)
tau_slot_sweep_by_intent = pd.concat(by_intent_frames, ignore_index=True).sort_values(['tau_slot', 'dataset_name', 'semantic_intent'], kind='mergesort').reset_index(drop=True)
tau_slot_sweep_by_slot = pd.concat(by_slot_frames, ignore_index=True).sort_values(['tau_slot', 'dataset_name', 'semantic_intent', 'semantic_slot'], kind='mergesort').reset_index(drop=True)

tau_slot_sweep_overall.to_csv(SWEEP_OVERALL_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)
tau_slot_sweep_by_dataset.to_csv(SWEEP_BY_DATASET_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)
tau_slot_sweep_by_intent.to_csv(SWEEP_BY_INTENT_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)
tau_slot_sweep_by_slot.to_csv(SWEEP_BY_SLOT_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)

display(tau_slot_sweep_overall)


In [ ]:
tau_slot_sweep_overall = tau_slot_sweep_overall.copy()
tau_slot_sweep_overall['feasible'] = tau_slot_sweep_overall['gold_slot_coverage'] >= COVERAGE_CONSTRAINT

best_feasible_df = tau_slot_sweep_overall.loc[tau_slot_sweep_overall['feasible']].sort_values(
    ['f1', 'precision', 'selective_accuracy', 'tau_slot'],
    ascending=[False, False, False, True],
    kind='mergesort',
).reset_index(drop=True)

if not best_feasible_df.empty:
    best_feasible_row = best_feasible_df.iloc[0].to_dict()
    best_infeasible_row = None
else:
    best_feasible_row = None
    best_infeasible_df = tau_slot_sweep_overall.sort_values(
        ['gold_slot_coverage', 'f1', 'precision', 'selective_accuracy', 'tau_slot'],
        ascending=[False, False, False, False, True],
        kind='mergesort',
    ).reset_index(drop=True)
    best_infeasible_row = best_infeasible_df.iloc[0].to_dict() if not best_infeasible_df.empty else None

best_tau_selection = {
    'run_id': RUN_ID,
    'generated_at': STUDY_TIMESTAMP,
    'selection_rule': {
        'objective': 'maximize_conll_f1',
        'coverage_constraint_metric': 'gold_slot_coverage',
        'coverage_constraint': COVERAGE_CONSTRAINT,
        'feasible_sort_order': ['f1 desc', 'precision desc', 'selective_accuracy desc', 'tau_slot asc'],
        'infeasible_sort_order': ['gold_slot_coverage desc', 'f1 desc', 'precision desc', 'selective_accuracy desc', 'tau_slot asc'],
    },
    'best_feasible_tau': to_serializable(best_feasible_row),
    'best_infeasible_tau': to_serializable(best_infeasible_row),
}

summary_row = best_tau_selection['best_feasible_tau'] or best_tau_selection['best_infeasible_tau']
written_best_prediction_paths = {'results_csv': []}
if summary_row:
    best_tau_slot = float(summary_row['tau_slot'])
    best_predictions_view = build_thresholded_prediction_view(slot_top1_cache, best_tau_slot)
    best_predictions_view.to_csv(BEST_PREDICTIONS_PATH, index=False, encoding='utf-8-sig', float_format=FLOAT_FORMAT)
    written_best_prediction_paths = export_evaluation_csvs(results_view=best_predictions_view, evaluation_root=EVALUATION_DIR)

study_config = {
    'run_id': RUN_ID,
    'generated_at': STUDY_TIMESTAMP,
    'repo_root': str(REPO_ROOT),
    'registry_path': str(REGISTRY_PATH),
    'sampling_seed': SAMPLING_SEED,
    'samples_per_intent': SAMPLES_PER_INTENT,
    'tau_grid': TAU_GRID,
    'slot_delimiter': SLOT_DELIMITER,
    'device': DEVICE,
    'spacy_batch_size': SPACY_BATCH_SIZE,
    'spacy_n_process': SPACY_N_PROCESS,
    'qa_batch_size': QA_BATCH_SIZE,
    'slot_sentence_batch_size': SLOT_SENTENCE_BATCH_SIZE,
    'nli_batch_size': NLI_BATCH_SIZE,
    'reuse_cached_slot_top1_scores': REUSE_CACHED_SLOT_TOP1_SCORES,
    'use_gold_intent_for_slots': True,
    'slot_score_basis': 'nli_plus_qa_bias',
    'slot_f1_method': 'conll_chunk_micro',
    'grouping_threshold': None,
    'dataset_paths': [str(path) for path in PUBLIC_CALIBRATION_DATASET_PATHS],
    'dataset_names': dataset_names,
    'raw_slot_specs': raw_slot_specs,
    'sample_counts_by_dataset_intent': sample_counts_by_dataset_intent.to_dict(orient='records'),
    'artifacts': {
        'sampled_dataset_view': str(SAMPLED_DATASET_PATH),
        'sampled_semantic_view': str(SAMPLED_SEMANTIC_PATH),
        'sample_counts': str(SAMPLE_COUNTS_PATH),
        'slot_top1_cache': str(SLOT_TOP1_CACHE_PATH),
        'tau_sweep_overall': str(SWEEP_OVERALL_PATH),
        'tau_sweep_by_dataset': str(SWEEP_BY_DATASET_PATH),
        'tau_sweep_by_intent': str(SWEEP_BY_INTENT_PATH),
        'tau_sweep_by_slot': str(SWEEP_BY_SLOT_PATH),
        'best_tau_selection': str(BEST_TAU_PATH),
        'best_predictions': str(BEST_PREDICTIONS_PATH),
    },
    'slot_inference_stats': slot_inference_stats,
    'model_snapshot': model_snapshot,
    'written_gold_paths': written_gold_paths,
    'written_top1_prediction_paths': written_top1_prediction_paths,
    'written_best_prediction_paths': written_best_prediction_paths,
}

with BEST_TAU_PATH.open('w', encoding='utf-8') as fp:
    json.dump(to_serializable(best_tau_selection), fp, ensure_ascii=False, indent=2)

with STUDY_CONFIG_PATH.open('w', encoding='utf-8') as fp:
    json.dump(to_serializable(study_config), fp, ensure_ascii=False, indent=2)

display(pd.DataFrame([summary_row]) if summary_row else pd.DataFrame())
print(f'Saved best tau slot selection to: {BEST_TAU_PATH}')
print(f'Saved study config to: {STUDY_CONFIG_PATH}')


## Uso esperado

1. Rodar as celulas em ordem.
2. Conferir `sample_counts_by_dataset_intent.csv` para validar a amostragem.
3. Conferir `public_calibration_pool_slot_top1_scores.csv` para o cache principal do sweep offline.
4. Conferir as colunas wide `{raw_slot} ({semantic_slot})`, `pred__...`, `score__...`.
5. Usar `tau_slot_sweep_overall.csv` e `best_tau_slot_selection.json` para escolher o threshold final; `f1`, `precision` e `recall` sao CoNLL/chunk micro.
